# Pinping a RunnableParallel with Other Runnables

This notebook requires the `grandalf` dependency

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [5]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books.
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers.
    Answer only by listing the projects.
    '''
)

chat_template_time = ChatPromptTemplate.from_template(
    '''
    I'm an intermediate level programmer.

    Considering the following literature:
    {books}

    Also, consider the following projects:
    {projects}

    Roughly how much time it would take me to complete the literature and the projects?
    '''
)

In [6]:
chat = ChatOpenAI(model='gpt-6-luna', seed=365, temperature=1, max_completion_tokens=15000)

In [8]:
string_parser = StrOutputParser()

In [9]:
chain_books = chat_template_books | chat | string_parser
chain_projects = chat_template_projects | chat | string_parser

In [10]:
chain_parallel = RunnableParallel({'books': chain_books,'projects': chain_projects})

In [ ]:
print(chain_parallel.invoke({'programming language': 'Python'}))

In [18]:
chain_time1 = (RunnableParallel({'books': chain_books,
                                'projects': chain_projects})
              | chat_template_time
              | chat
              | string_parser
             )

In [19]:
chain_time2 = ({'books': chain_books,'projects': chain_projects}
              | chat_template_time
              | chat
              | string_parser
             )

In [20]:
print(chain_time2.invoke({'programming language': 'Python'}))

A reasonable estimate is **about 300–500 focused hours** to read the books thoughtfully and build a working first version of all three projects. The range depends mainly on how much you experiment with the code and how polished you want the projects to be.

| Work | Rough effort |
|---|---:|
| *Fluent Python* | 70–120 hours |
| *Effective Python* | 25–45 hours |
| *Python Cookbook* | 50–90 hours |
| Personal Finance Dashboard (MVP) | 40–80 hours |
| Local Event Finder (MVP) | 30–60 hours |
| Document Search Engine (MVP) | 50–100 hours |
| **Total** | **265–495 hours** |

These are estimates for an intermediate programmer, including trying examples, writing and debugging project code, and basic testing. The project estimates assume useful prototypes—not production-ready apps with extensive security, deployment, monitoring, or polished interfaces.

At a steady pace, that works out to roughly:

- **5 hours/week:** 12–23 months  
- **10 hours/week:** 6–12 months  
- **15 hours/week:** 4–8 

In [21]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            